In [1]:
from pprint import pprint

from vaxflux import (
    Implementation,
    Intervention,
    LogisticCurve,
    PartiallyPooledGaussianCovariate,
    VaxfluxModel,
)
from vaxflux.covariates import CovariateCategories
from vaxflux.dates import (
    SeasonRange,
    daily_date_ranges,
)

In [2]:
seasons = [
    SeasonRange(season="2022/23", start_date="2022-10-03", end_date="2023-02-05"),
    SeasonRange(season="2023/24", start_date="2023-10-02", end_date="2024-02-04"),
    SeasonRange(season="2024/25", start_date="2024-10-07", end_date="2025-02-02"),
]
dates = daily_date_ranges(seasons, range_days=6)

In [3]:
age_covariate_categories = CovariateCategories(
    covariate="age",
    categories=["youth", "adult", "elderly"],
)

In [4]:
covariates = [
    PartiallyPooledGaussianCovariate(
        parameter="m",
        covariate=None,
        mu=(0.5, 0.25),
        sigma=0.5,
    ),
    PartiallyPooledGaussianCovariate(
        parameter="r",
        covariate=None,
        mu=(-2.5, 1.0),
        sigma=1.0,
    ),
    PartiallyPooledGaussianCovariate(
        parameter="s",
        covariate=None,
        mu=(40.0, 10.0),
        sigma=10.0,
    ),
    PartiallyPooledGaussianCovariate(
        parameter="m",
        covariate="age",
        mu=(0.25, 0.25),
        sigma=0.25,
    ),
]

In [5]:
interventions = [
    Intervention(
        name="tv_ads",
        parameter="m",
        distribution="Normal",
        distribution_kwargs={"loc": 0.0, "scale": 0.1},
    ),
]
implementations = [
    Implementation(
        intervention="tv_ads",
        season="2023/24",
        start_date="2023-10-15",
        end_date="2023-11-15",
        covariate_categories={"age": "adult"},
    ),
    Implementation(
        intervention="tv_ads",
        season="2024/25",
        start_date="2024-11-01",
        end_date="2024-12-01",
        covariate_categories=None,
    ),
]

In [6]:
import pandas as pd

observations = pd.DataFrame.from_records(
    [
        {
            "season": "2023/24",
            "start_date": "2023-10-16",
            "end_date": "2023-10-22",
            "type": "incidence",
            "value": 0.05,
            "age": "adult",
        },
        {
            "season": "2023/24",
            "start_date": "2023-10-23",
            "end_date": "2023-10-29",
            "type": "incidence",
            "value": 0.07,
            "age": "adult",
        },
    ]
)

In [7]:
curve = LogisticCurve()
model = (
    VaxfluxModel(curve=curve)
    .add_seasons(seasons)
    .add_dates(dates)
    .add_covariate_categories(age_covariate_categories)
    .add_covariates(covariates)
    .add_interventions(interventions)
    .add_implementations(implementations)
    .add_observations(observations)
    .add_observation_process(kind="normal", noise=0.02)
)
model

In [8]:
prior_predictive = model.prior_predictive(samples=150)
pprint(list(prior_predictive.keys()))

['covariate_values_m_age',
 'covariate_values_m_season',
 'covariate_values_r_season',
 'covariate_values_s_season',
 'incidence_2022_23_age_adult_2022_10_03_2022_10_09',
 'incidence_2022_23_age_adult_2022_10_10_2022_10_16',
 'incidence_2022_23_age_adult_2022_10_17_2022_10_23',
 'incidence_2022_23_age_adult_2022_10_24_2022_10_30',
 'incidence_2022_23_age_adult_2022_10_31_2022_11_06',
 'incidence_2022_23_age_adult_2022_11_07_2022_11_13',
 'incidence_2022_23_age_adult_2022_11_14_2022_11_20',
 'incidence_2022_23_age_adult_2022_11_21_2022_11_27',
 'incidence_2022_23_age_adult_2022_11_28_2022_12_04',
 'incidence_2022_23_age_adult_2022_12_05_2022_12_11',
 'incidence_2022_23_age_adult_2022_12_12_2022_12_18',
 'incidence_2022_23_age_adult_2022_12_19_2022_12_25',
 'incidence_2022_23_age_adult_2022_12_26_2023_01_01',
 'incidence_2022_23_age_adult_2023_01_02_2023_01_08',
 'incidence_2022_23_age_adult_2023_01_09_2023_01_15',
 'incidence_2022_23_age_adult_2023_01_16_2023_01_22',
 'incidence_2022_23